In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/energy-anomaly-detection/sample_submission.csv
/kaggle/input/competitions/energy-anomaly-detection/train.csv
/kaggle/input/competitions/energy-anomaly-detection/test.csv
/kaggle/input/competitions/energy-anomaly-detection/train_features.csv
/kaggle/input/competitions/energy-anomaly-detection/test_features.csv


In [4]:
df = pd.read_csv("/kaggle/input/competitions/energy-anomaly-detection/train.csv", delimiter=',', encoding='utf8')
pd.set_option('display.max_columns', None)
df.tail()

,building_id,timestamp,meter_reading,anomaly
1749489,1316,2016-12-31 23:00:00,38.844,0
1749490,1318,2016-12-31 23:00:00,202.893,0
1749491,1319,2016-12-31 23:00:00,NaN,0
1749492,1323,2016-12-31 23:00:00,172.000,0
1749493,1353,2016-12-31 23:00:00,2.400,0


In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.tail()

,building_id,timestamp,meter_reading,anomaly
1749489,1316,2016-12-31 23:00:00,38.844,0
1749490,1318,2016-12-31 23:00:00,202.893,0
1749491,1319,2016-12-31 23:00:00,NaN,0
1749492,1323,2016-12-31 23:00:00,172.000,0
1749493,1353,2016-12-31 23:00:00,2.400,0


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1749494 entries, 0 to 1749493
Data columns (total 4 columns):
 #   Column         Dtype         
---  ------         -----         
 0   building_id    int64         
 1   timestamp      datetime64[ns]
 2   meter_reading  float64       
 3   anomaly        int64         
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 53.4 MB


In [6]:
df['building_id'].unique()

array([   1,   32,   41,   55,   69,   79,   82,   91,  107,  108,  111,
        112,  117,  118,  119,  121,  136,  137,  139,  141,  144,  147,
        148,  149,  159,  171,  173,  174,  181,  183,  190,  235,  238,
        240,  246,  247,  248,  253,  254,  263,  270,  275,  276,  278,
        290,  293,  312,  318,  335,  345,  356,  423,  439,  492,  534,
        560,  623,  653,  657,  658,  666,  667,  673,  675,  677,  680,
        683,  685,  687,  693,  697,  698,  701,  708,  710,  721,  722,
        729,  730,  732,  739,  742,  801,  827,  844,  848,  879,  880,
        881,  882,  884,  886,  887,  889,  890,  892,  893,  894,  895,
        896,  903,  905,  909,  914,  919,  922,  924,  925,  926,  928,
        929,  931,  935,  936,  941,  942,  945,  948,  950,  952,  961,
        966,  967,  968,  969,  970,  971,  973,  974,  975,  977,  978,
        981,  988,  990,  992,  994,  996, 1001, 1007, 1068, 1073, 1074,
       1106, 1120, 1128, 1137, 1141, 1143, 1147, 11

In [6]:
d1=df[df['building_id']==118]

In [8]:
d1.isnull().sum()

building_id      0
timestamp        0
meter_reading    0
anomaly          0
dtype: int64

In [9]:
d1['anomaly'].value_counts()

anomaly
0    8618
1     166
Name: count, dtype: int64

In [7]:
d1=df[df['building_id']==118]
hourly_df=d1.drop(['building_id'],axis=1)
hourly_df = hourly_df.sort_values('timestamp').reset_index(drop=True)

In [41]:
hourly_df

,timestamp,meter_reading,anomaly
0,2016-01-01 00:00:00,117.2,0
1,2016-01-01 01:00:00,234.4,0
2,2016-01-01 02:00:00,234.8,0
3,2016-01-01 03:00:00,236.4,0
4,2016-01-01 04:00:00,239.8,0
...,...,...,...
8779,2016-12-31 19:00:00,321.8,0
8780,2016-12-31 20:00:00,299.0,0
8781,2016-12-31 21:00:00,286.1,0
8782,2016-12-31 22:00:00,280.7,0


In [8]:
hourly_df['day'] = hourly_df['timestamp'].dt.day
hourly_df['day_name'] = hourly_df['timestamp'].dt.day_name()
hourly_df['month'] = hourly_df['timestamp'].dt.month
hourly_df['day_of_week'] = hourly_df['timestamp'].dt.dayofweek
hourly_df['day_of_year'] = hourly_df['timestamp'].dt.dayofyear
hourly_df['week_of_year'] = hourly_df['timestamp'].dt.isocalendar().week
hourly_df['hour'] = hourly_df['timestamp'].dt.hour
hourly_df['is_weekend'] = hourly_df['day_name'].isin(['Saturday','Sunday']).astype(int)
hourly_df['usage_change'] = hourly_df['meter_reading'].diff().fillna(0)
hourly_df['rolling_mean_3'] = hourly_df['meter_reading'].rolling(3).mean().fillna(method='bfill')
hourly_df['rolling_std_3'] = hourly_df['meter_reading'].rolling(3).std().fillna(method='bfill')
hourly_df['usage_diff_3h'] = hourly_df['meter_reading'].diff(3).fillna(0)

/tmp/ipykernel_57/2863443667.py:10: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hourly_df['rolling_mean_3'] = hourly_df['meter_reading'].rolling(3).mean().fillna(method='bfill')
/tmp/ipykernel_57/2863443667.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hourly_df['rolling_std_3'] = hourly_df['meter_reading'].rolling(3).std().fillna(method='bfill')


In [9]:
hourly_df

,timestamp,meter_reading,anomaly,day,day_name,month,day_of_week,day_of_year,week_of_year,hour,is_weekend,usage_change,rolling_mean_3,rolling_std_3,usage_diff_3h
0,2016-01-01 00:00:00,117.2,0,1,Friday,1,4,1,53,0,0,0.0,195.466667,67.781217,0.0
1,2016-01-01 01:00:00,234.4,0,1,Friday,1,4,1,53,1,0,117.2,195.466667,67.781217,0.0
2,2016-01-01 02:00:00,234.8,0,1,Friday,1,4,1,53,2,0,0.4,195.466667,67.781217,0.0
3,2016-01-01 03:00:00,236.4,0,1,Friday,1,4,1,53,3,0,1.6,235.200000,1.058301,119.2
4,2016-01-01 04:00:00,239.8,0,1,Friday,1,4,1,53,4,0,3.4,237.000000,2.553429,5.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,2016-12-31 19:00:00,321.8,0,31,Saturday,12,5,366,52,19,1,-2.2,324.800000,3.469870,-9.6
8780,2016-12-31 20:00:00,299.0,0,31,Saturday,12,5,366,52,20,1,-22.8,314.933333,13.842447,-29.6
8781,2016-12-31 21:00:00,286.1,0,31,Saturday,12,5,366,52,21,1,-12.9,302.300000,18.077334,-37.9
8782,2016-12-31 22:00:00,280.7,0,31,Saturday,12,5,366,52,22,1,-5.4,288.600000,9.402659,-41.1


#train and fine

In [27]:
# Unsupervised Continual Anomaly Detection
# Autoencoder + Isolation Forest (with Save & Load)

import numpy as np
import pandas as pd
import os
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.ensemble import IsolationForest

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam


# ================= CONFIG =================
FEATURES = [
    'meter_reading',
    'hour',
    'day',
    'day_of_week',
    'day_of_year',
    'week_of_year',
    'month',
    'is_weekend',
    'usage_change',
    'rolling_mean_3',
    'rolling_std_3',
    'usage_diff_3h'
]

LATENT_DIM = 16
EPOCHS_INIT = 40
EPOCHS_CONT = 10
BATCH_SIZE = 256
BUFFER_SIZE = 5000

# Save paths
MODEL_PATH = "ae_model.h5"
ISO_PATH = "iso.pkl"
SCALER_PATH = "scaler.pkl"
BUFFER_PATH = "buffer.npy"


# ================= DATA PREP =================
hourly_df['timestamp'] = pd.to_datetime(hourly_df['timestamp'])
hourly_df = hourly_df.sort_values('timestamp').reset_index(drop=True)



hourly_df['month_idx'] = hourly_df['timestamp'].dt.month

train_df = hourly_df[hourly_df['month_idx'] <= 6]
continual_df = hourly_df[hourly_df['month_idx'] > 6]
# ===== SCALER HANDLING =====
if os.path.exists(SCALER_PATH):
    scaler = joblib.load(SCALER_PATH)

    # Apply only transform (NO fit)
    train_df[FEATURES] = scaler.transform(train_df[FEATURES])

else:
    scaler = MinMaxScaler()
    train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES])

    joblib.dump(scaler, SCALER_PATH)

X_train = train_df[FEATURES].values


# ================= REPLAY BUFFER =================
BUFFER_SIZE = min(BUFFER_SIZE, len(X_train))
buffer_X = X_train[np.random.choice(len(X_train), BUFFER_SIZE, replace=False)]


# ================= AUTOENCODER =================
def build_autoencoder(input_dim, latent_dim):
    inp = Input(shape=(input_dim,))
    x = Dense(64, activation='relu')(inp)
    x = Dense(32, activation='relu')(x)
    z = Dense(latent_dim, activation='relu')(x)

    x = Dense(32, activation='relu')(z)
    x = Dense(64, activation='relu')(x)
    out = Dense(input_dim, activation='sigmoid')(x)

    model = Model(inp, out)
    model.compile(optimizer=Adam(1e-3), loss='mse')
    return model


# ================= LOAD OR TRAIN =================
if os.path.exists(MODEL_PATH):
    print(" Loading existing model...")

    model = load_model(MODEL_PATH, compile=False)
    model.compile(optimizer=Adam(1e-3), loss='mse')
    iso = joblib.load(ISO_PATH)
    scaler = joblib.load(SCALER_PATH)
    buffer_X = np.load(BUFFER_PATH)

else:
    print(" Training from scratch...")

    model = build_autoencoder(len(FEATURES), LATENT_DIM)

    model.fit(
        X_train, X_train,
        epochs=EPOCHS_INIT,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1
    )

    # Build encoder
    encoder = Model(
        inputs=model.input,
        outputs=model.layers[3].output
    )

    # Train Isolation Forest
    Z_train = encoder.predict(X_train, verbose=0)

    iso = IsolationForest(
        n_estimators=300,
        max_samples='auto',
        contamination=0.03,
        random_state=42
    )
    iso.fit(Z_train)

    # Save everything
    model.save(MODEL_PATH)
    joblib.dump(iso, ISO_PATH)
    joblib.dump(scaler, SCALER_PATH)
    np.save(BUFFER_PATH, buffer_X)


# ================= ALWAYS BUILD ENCODER =================
encoder = Model(
    inputs=model.input,
    outputs=model.layers[3].output
)


# ================= CONTINUAL LOOP =================
results = {}

for m in sorted(continual_df['month_idx'].unique()):
    print(f"\n================ MONTH {m} ================")

    month_df = continual_df[continual_df['month_idx'] == m]
    X_month = scaler.transform(month_df[FEATURES])
    y_true = month_df['anomaly'].values  # only for evaluation

    # ----- LATENT REPRESENTATION -----
    Z_month = encoder.predict(X_month, verbose=0)

    # ----- PREDICTION -----
    y_pred = (iso.predict(Z_month) == -1).astype(int)
    scores = -iso.decision_function(Z_month)

    # ----- METRICS -----
    if len(np.unique(y_true)) > 1:
        roc = roc_auc_score(y_true, scores)
        pr = average_precision_score(y_true, scores)
        print(f"ROC-AUC: {roc:.4f}")
        print(f"PR-AUC : {pr:.4f}")
    else:
        roc, pr = np.nan, np.nan
        print("Only one class present in y_true")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    results[m] = {'roc_auc': roc, 'pr_auc': pr}

    # ================= CONTINUAL UPDATE =================

    # Update buffer
    buffer_X = np.vstack([buffer_X, X_month])
    if len(buffer_X) > BUFFER_SIZE:
        buffer_X = buffer_X[-BUFFER_SIZE:]

    # Retrain Autoencoder
    model.fit(
        buffer_X, buffer_X,
        epochs=EPOCHS_CONT,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=0
    )

    # Update encoder
    encoder = Model(
        inputs=model.input,
        outputs=model.layers[3].output
    )

    # Retrain Isolation Forest
    Z_buffer = encoder.predict(buffer_X, verbose=0)
    iso.fit(Z_buffer)

    # ================= SAVE AFTER EACH STEP =================
    model.save(MODEL_PATH)
    joblib.dump(iso, ISO_PATH)
    np.save(BUFFER_PATH, buffer_X)


# ================= SUMMARY =================
print("\n===== CONTINUAL SUMMARY =====")
for m, v in results.items():
    print(f"Month {m}: {v}")

 Training from scratch...
Epoch 1/40


/tmp/ipykernel_57/2171454189.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES])


18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0881
Epoch 2/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0739 
Epoch 3/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0621 
Epoch 4/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0477 
Epoch 5/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0326 
Epoch 6/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0224 
Epoch 7/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0191 
Epoch 8/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0162 
Epoch 9/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0128 
Epoch 10/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0095 
Epoch 11/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0074 
Epoch 12/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0055 
Epoch 13/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0048 
Epoch 14/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0042 
Epoch 15/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0036 
Epoch 16/40
18/1


================ MONTH 7 ================
Only one class present in y_true

Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.9960    0.9980       744
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.9960       744
   macro avg     0.5000    0.4980    0.4990       744
weighted avg     1.0000    0.9960    0.9980       744




================ MONTH 8 ================
Only one class present in y_true

Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.9973    0.9987       744
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.9973       744
   macro avg     0.5000    0.4987    0.4993       744
weighted avg     1.0000    0.9973    0.9987       744




================ MONTH 9 ================
ROC-AUC: 0.7857
PR-AUC : 0.1418

Classification report:
              precision    recall  f1-score   support

           0     0.9944    0.9944    0.9944       715
           1     0.2000    0.2000    0.2000         5

    accuracy                         0.9889       720
   macro avg     0.5972    0.5972    0.5972       720
weighted avg     0.9889    0.9889    0.9889       720




================ MONTH 10 ================
ROC-AUC: 0.8639
PR-AUC : 0.0155

Classification report:
              precision    recall  f1-score   support

           0     0.9972    0.9704    0.9836       742
           1     0.0000    0.0000    0.0000         2

    accuracy                         0.9677       744
   macro avg     0.4986    0.4852    0.4918       744
weighted avg     0.9945    0.9677    0.9810       744




================ MONTH 11 ================
ROC-AUC: 0.9624
PR-AUC : 0.5801

Classification report:
              precision    recall  f1-score   support

           0     0.9862    0.9625    0.9742       666
           1     0.6429    0.8333    0.7258        54

    accuracy                         0.9528       720
   macro avg     0.8145    0.8979    0.8500       720
weighted avg     0.9604    0.9528    0.9555       720




================ MONTH 12 ================
ROC-AUC: 0.9866
PR-AUC : 0.8689

Classification report:
              precision    recall  f1-score   support

           0     0.9703    0.9957    0.9828       690
           1     0.9167    0.6111    0.7333        54

    accuracy                         0.9677       744
   macro avg     0.9435    0.8034    0.8581       744
weighted avg     0.9664    0.9677    0.9647       744




===== CONTINUAL SUMMARY =====
Month 7: {'roc_auc': nan, 'pr_auc': nan}
Month 8: {'roc_auc': nan, 'pr_auc': nan}
Month 9: {'roc_auc': np.float64(0.7857342657342656), 'pr_auc': np.float64(0.14177495142201024)}
Month 10: {'roc_auc': np.float64(0.8638814016172507), 'pr_auc': np.float64(0.01552868658131816)}
Month 11: {'roc_auc': np.float64(0.9624068512957401), 'pr_auc': np.float64(0.5800744978488057)}
Month 12: {'roc_auc': np.float64(0.9865539452495974), 'pr_auc': np.float64(0.8689334473182191)}


#load and fine 

In [11]:
# Unsupervised Continual Anomaly Detection
# Autoencoder + Isolation Forest (with Save & Load)

import numpy as np
import pandas as pd
import os
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.ensemble import IsolationForest

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam


# ================= CONFIG =================
FEATURES = [
    'meter_reading',
    'hour',
    'day',
    'day_of_week',
    'day_of_year',
    'week_of_year',
    'month',
    'is_weekend',
    'usage_change',
    'rolling_mean_3',
    'rolling_std_3',
    'usage_diff_3h'
]

LATENT_DIM = 16
EPOCHS_INIT = 40
EPOCHS_CONT = 10
BATCH_SIZE = 256
BUFFER_SIZE = 5000

# Save paths
MODEL_PATH = "ae_model.h5"
ISO_PATH = "iso.pkl"
SCALER_PATH = "scaler.pkl"
BUFFER_PATH = "buffer.npy"


# ================= DATA PREP =================
hourly_df['timestamp'] = pd.to_datetime(hourly_df['timestamp'])
hourly_df = hourly_df.sort_values('timestamp').reset_index(drop=True)



hourly_df['month_idx'] = hourly_df['timestamp'].dt.month

train_df = hourly_df[hourly_df['month_idx'] <= 6]
continual_df = hourly_df[hourly_df['month_idx'] > 6]
# ===== SCALER HANDLING =====
if os.path.exists(SCALER_PATH):
    scaler = joblib.load(SCALER_PATH)

    # Apply only transform (NO fit)
    train_df[FEATURES] = scaler.transform(train_df[FEATURES])

else:
    scaler = MinMaxScaler()
    train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES])

    joblib.dump(scaler, SCALER_PATH)

X_train = train_df[FEATURES].values


# ================= REPLAY BUFFER =================
BUFFER_SIZE = min(BUFFER_SIZE, len(X_train))
buffer_X = X_train[np.random.choice(len(X_train), BUFFER_SIZE, replace=False)]


# ================= AUTOENCODER =================
def build_autoencoder(input_dim, latent_dim):
    inp = Input(shape=(input_dim,))
    x = Dense(64, activation='relu')(inp)
    x = Dense(32, activation='relu')(x)
    z = Dense(latent_dim, activation='relu')(x)

    x = Dense(32, activation='relu')(z)
    x = Dense(64, activation='relu')(x)
    out = Dense(input_dim, activation='sigmoid')(x)

    model = Model(inp, out)
    model.compile(optimizer=Adam(1e-3), loss='mse')
    return model


# ================= LOAD OR TRAIN =================
if os.path.exists(MODEL_PATH):
    print(" Loading existing model...")

    model = load_model(MODEL_PATH, compile=False)
    model.compile(optimizer=Adam(1e-3), loss='mse')
    iso = joblib.load(ISO_PATH)
    scaler = joblib.load(SCALER_PATH)
    buffer_X = np.load(BUFFER_PATH)

else:
    print(" Training from scratch...")

    model = build_autoencoder(len(FEATURES), LATENT_DIM)

    model.fit(
        X_train, X_train,
        epochs=EPOCHS_INIT,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1
    )

    # Build encoder
    encoder = Model(
        inputs=model.input,
        outputs=model.layers[3].output
    )

    # Train Isolation Forest
    Z_train = encoder.predict(X_train, verbose=0)

    iso = IsolationForest(
        n_estimators=300,
        max_samples='auto',
        contamination=0.03,
        random_state=42
    )
    iso.fit(Z_train)

    # Save everything
    model.save(MODEL_PATH)
    joblib.dump(iso, ISO_PATH)
    joblib.dump(scaler, SCALER_PATH)
    np.save(BUFFER_PATH, buffer_X)


# ================= ALWAYS BUILD ENCODER =================
encoder = Model(
    inputs=model.input,
    outputs=model.layers[3].output
)


# ================= CONTINUAL LOOP =================
results = {}

for m in sorted(continual_df['month_idx'].unique()):
    print(f"\n================ MONTH {m} ================")

    month_df = continual_df[continual_df['month_idx'] == m]
    X_month = scaler.transform(month_df[FEATURES])
    y_true = month_df['anomaly'].values  # only for evaluation

    # ----- LATENT REPRESENTATION -----
    Z_month = encoder.predict(X_month, verbose=0)

    # ----- PREDICTION -----
    y_pred = (iso.predict(Z_month) == -1).astype(int)
    scores = -iso.decision_function(Z_month)

    # ----- METRICS -----
    if len(np.unique(y_true)) > 1:
        roc = roc_auc_score(y_true, scores)
        pr = average_precision_score(y_true, scores)
        print(f"ROC-AUC: {roc:.4f}")
        print(f"PR-AUC : {pr:.4f}")
    else:
        roc, pr = np.nan, np.nan
        print("Only one class present in y_true")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    results[m] = {'roc_auc': roc, 'pr_auc': pr}

    # ================= CONTINUAL UPDATE =================

    # Update buffer
    buffer_X = np.vstack([buffer_X, X_month])
    if len(buffer_X) > BUFFER_SIZE:
        buffer_X = buffer_X[-BUFFER_SIZE:]

    # Retrain Autoencoder
    model.fit(
        buffer_X, buffer_X,
        epochs=EPOCHS_CONT,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=0
    )

    # Update encoder
    encoder = Model(
        inputs=model.input,
        outputs=model.layers[3].output
    )

    # Retrain Isolation Forest
    Z_buffer = encoder.predict(buffer_X, verbose=0)
    iso.fit(Z_buffer)

    # ================= SAVE AFTER EACH STEP =================
    model.save(MODEL_PATH)
    joblib.dump(iso, ISO_PATH)
    np.save(BUFFER_PATH, buffer_X)


# ================= SUMMARY =================
print("\n===== CONTINUAL SUMMARY =====")
for m, v in results.items():
    print(f"Month {m}: {v}")

/tmp/ipykernel_57/2518235097.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[FEATURES] = scaler.transform(train_df[FEATURES])


 Loading existing model...

================ MONTH 7 ================
Only one class present in y_true

Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.9731    0.9864       744
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.9731       744
   macro avg     0.5000    0.4866    0.4932       744
weighted avg     1.0000    0.9731    0.9864       744




================ MONTH 8 ================
Only one class present in y_true

Classification report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       744

    accuracy                         1.0000       744
   macro avg     1.0000    1.0000    1.0000       744
weighted avg     1.0000    1.0000    1.0000       744




================ MONTH 9 ================
ROC-AUC: 0.8487
PR-AUC : 0.0875

Classification report:
              precision    recall  f1-score   support

           0     0.9931    1.0000    0.9965       715
           1     0.0000    0.0000    0.0000         5

    accuracy                         0.9931       720
   macro avg     0.4965    0.5000    0.4983       720
weighted avg     0.9862    0.9931    0.9896       720




================ MONTH 10 ================
ROC-AUC: 0.9960
PR-AUC : 0.6250

Classification report:
              precision    recall  f1-score   support

           0     0.9987    0.9973    0.9980       742
           1     0.3333    0.5000    0.4000         2

    accuracy                         0.9960       744
   macro avg     0.6660    0.7487    0.6990       744
weighted avg     0.9969    0.9960    0.9964       744




================ MONTH 11 ================
ROC-AUC: 0.9986
PR-AUC : 0.9879

Classification report:
              precision    recall  f1-score   support

           0     0.9970    0.9970    0.9970       666
           1     0.9630    0.9630    0.9630        54

    accuracy                         0.9944       720
   macro avg     0.9800    0.9800    0.9800       720
weighted avg     0.9944    0.9944    0.9944       720




================ MONTH 12 ================
ROC-AUC: 0.9986
PR-AUC : 0.9879

Classification report:
              precision    recall  f1-score   support

           0     0.9971    0.9899    0.9935       690
           1     0.8814    0.9630    0.9204        54

    accuracy                         0.9879       744
   macro avg     0.9392    0.9764    0.9569       744
weighted avg     0.9887    0.9879    0.9881       744




===== CONTINUAL SUMMARY =====
Month 7: {'roc_auc': nan, 'pr_auc': nan}
Month 8: {'roc_auc': nan, 'pr_auc': nan}
Month 9: {'roc_auc': np.float64(0.8486713286713287), 'pr_auc': np.float64(0.08746639849004104)}
Month 10: {'roc_auc': np.float64(0.9959568733153639), 'pr_auc': np.float64(0.625)}
Month 11: {'roc_auc': np.float64(0.9986375264153042), 'pr_auc': np.float64(0.9879491555203255)}
Month 12: {'roc_auc': np.float64(0.9986044015029522), 'pr_auc': np.float64(0.9879253857535405)}
